In [1]:
import gdsfactory as gf
from gdsfactory.generic_tech import get_generic_pdk

# 1. Activate the Generic PDK
# In a notebook, this ensures the standard layers are loaded
gf.config.rich_output()
PDK = get_generic_pdk()
PDK.activate()

@gf.cell
def transceiver_reference():
    """
    Sample Optical Transceiver Layout (Tx Side)
    Includes:
    - Optical: Grating Couplers -> MZI Modulator
    - Electrical: MZI Heater -> Bondpads
    """
    c = gf.Component("transceiver_sample")

    # --- COMPONENTS ---

    # 1. MZI Modulator (with top metal heater)
    # This component has optical ports ('o1', 'o2') and electrical ports ('e1', 'e2')
    mzi = c << gf.components.mzi_phase_shifter_top_heater_metal(
        length_x=200,  # Length of the phase shifter arm
        delta_length=20, # Path length difference
    )
    mzi.x = 0
    mzi.y = 0

    # 2. Optical I/O: Grating Couplers
    # 127um pitch is standard for fiber arrays
    gc_in = c << gf.components.grating_coupler_elliptical_te()
    gc_out = c << gf.components.grating_coupler_elliptical_te()
    
    gc_in.rotate(180) # Face left for input
    gc_out.rotate(180) # Face left for output

    # Position GCs relative to the MZI
    gc_in.x = mzi.xmin - 300
    gc_in.y = mzi.y - 63.5
    gc_out.x = mzi.xmin - 300
    gc_out.y = mzi.y + 63.5

    # 3. Electrical I/O: Bondpads
    pad_top = c << gf.components.pad(size=(80, 80))
    pad_bot = c << gf.components.pad(size=(80, 80))

    # Place pads above and below the MZI
    pad_top.x = mzi.x
    pad_top.ymin = mzi.ymax + 100
    
    pad_bot.x = mzi.x
    pad_bot.ymax = mzi.ymin - 100

    # --- OPTICAL ROUTING ---
    
    # Route input GC to MZI input (o1)
    route_in = gf.routing.get_route(
        gc_in.ports["o1"],
        mzi.ports["o1"],
        cross_section="strip"
    )
    c.add(route_in.references)

    # Route MZI output (o2) to output GC
    route_out = gf.routing.get_route(
        mzi.ports["o2"],
        gc_out.ports["o1"],
        cross_section="strip"
    )
    c.add(route_out.references)


    # --- ELECTRICAL ROUTING ---
    
    # Route Top Pad to MZI Top Heater Port (e2)
    # Uses metal cross-section automatically via get_route_electrical
    route_elec_top = gf.routing.get_route_electrical(
        mzi.ports["e2"],
        pad_top.ports["e4"], # Connect to bottom of top pad
        bend="bend_euler",
    )
    c.add(route_elec_top.references)

    # Route Bottom Pad to MZI Bottom Heater Port (e1)
    route_elec_bot = gf.routing.get_route_electrical(
        mzi.ports["e1"],
        pad_bot.ports["e2"], # Connect to top of bottom pad
        bend="bend_euler",
    )
    c.add(route_elec_bot.references)

    return c

# --- EXECUTION ---

# Generate the component
c = transceiver_reference()

# Save GDS file (outputs to the same folder as the notebook)
gds_path = "transceiver_reference.gds"
c.write_gds(gds_path)
print(f"Layout saved to: {gds_path}")

# Plot directly in the notebook
# This renders the interactive layout viewer
c.plot()

AttributeError: module 'gdsfactory.routing' has no attribute 'get_route'